In [6]:
# Skriptas pateikia sąraša 'švarių' (vizualiai atrinktų ir su sužymėtais triukšmais) įrašų modelių apmokymui.
# Sarašas sudaromas panaudojant failus iš aplanko 1_PREPARE_TRAIN_UNET_DATA/ecg_zive_npy_for_preparing.

from __future__ import annotations

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, TypedDict, Any
from collections import Counter
import pandas as pd
import sys

def _display_compact_table(
    df: pd.DataFrame,
    title: str = "",
    col_widths_px: Optional[Dict[str, int]] = None,
    default_px: int = 90,
) -> None:
    """
    Jupyter: render df as compact HTML table with fixed column widths + ellipsis.
    Fallback: print a compact plain-text table.
    """
    if title:
        print(f"\n=== {title} ===")

    if df is None or df.empty:
        print("(empty)")
        return

    col_widths_px = col_widths_px or {}
    widths = {c: f"{int(col_widths_px.get(c, default_px))}px" for c in df.columns}

    try:
        from IPython.display import display

        base_styles = [
            {"selector": "table", "props": [("table-layout", "fixed"), ("width", "auto")]},
            {"selector": "th, td", "props": [
                ("padding", "2px 6px"),
                ("font-size", "12px"),
                ("white-space", "nowrap"),
                ("overflow", "hidden"),
                ("text-overflow", "ellipsis"),
                ("vertical-align", "top"),
            ]},
        ]

        col_styles = [
            {"selector": f"th.col{i}, td.col{i}", "props": [("width", widths[c])]}
            for i, c in enumerate(df.columns)
        ]

        display(df.style.set_table_styles(base_styles + col_styles))

    except Exception:
        # Fallback: truncate long values for readability in terminals/logs
        def _truncate(x: Any, n: int) -> str:
            s = "" if x is None else str(x)
            return s if len(s) <= n else s[: n - 1] + "…"

        df_show = df.copy()
        # rough px->chars mapping
        char_limits = {c: max(8, int((col_widths_px.get(c, default_px) / 7))) for c in df_show.columns}
        for c in df_show.columns:
            df_show[c] = df_show[c].map(lambda v, n=char_limits[c]: _truncate(v, n))

        print(df_show.to_string(index=False))

# Compact column widths for df_meta (tune if needed)
META_COL_WIDTHS_PX: Dict[str, int] = {
    "filename": 90,
    "length": 70,
    "quality": 70,
    "noni": 70,
    "tag": 60,
    "mark": 60,
    "N": 55,
    "S": 55,
    "V": 55,
    "U": 55,
    "recordingId": 170,
    "basename": 90,
    "userId": 170,
    "comment": 260,   # long text -> ellipsis keeps it compact
}

def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    print("\nSkaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    print("Įrašų sąraše:", len(names))
    return names, df


class EcgJsonSummary(TypedDict):
    # 0/1 presence for each known flag
    flags_01: Dict[str, int]

    # counts derived from noises_annotated
    noises_annotated_count: int          # valid intervals (parsed successfully)
    noises_annotated_raw_count: int      # raw list length from JSON (may include broken entries)

    # optional: keep parsed intervals if you still want them
    intervals: List[Tuple[int, int]]


from typing import Dict, List, Optional, Any

def print_ecg_json_summary_row(
    summary: Dict[str, Any],
    known_flags: Optional[List[str]] = None,
    prefix: str = "",
) -> None:
    """
    Prints a compact single-line 'row' showing:
      - the names of flags that are set to 1 (non-zero)
      - noises_annotated_count

    Expected `summary` shape:
      {
        "flags_01": {"PSEUDO_ANNOTATED": 0/1, ...},
        "noises_annotated_count": int,
        ...
      }
    """
    known_flags = known_flags or [
        "PSEUDO_ANNOTATED",
        "FOR_EXTERNAL_ANNOTATING",
        "EXTERNALLY_ANNOTATED",
        "FULLY_ANNOTATED_PROFESSIONALLY",
    ]

    flags_01 = summary.get("flags_01") or {}
    if not isinstance(flags_01, dict):
        flags_01 = {}

    # keep order as in known_flags
    active_flags = [f for f in known_flags if int(flags_01.get(f, 0)) != 0]

    noises_cnt = int(summary.get("noises_annotated_count", 0))

    # Print a single compact row
    flags_str = ", ".join(active_flags) if active_flags else "-"
    print(f"{prefix}FLAGS: {flags_str} | noises_annotated_count: {noises_cnt}")


def count_flags_from_ecg_json(
    json_path: Path,
    known_flags: Optional[List[str]] = None,
    return_intervals: bool = False,
) -> Optional[EcgJsonSummary]:
    """
    Returns a summary of JSON:
      - flags presence as 0/1 per flag type (based on known_flags)
      - count of noises_annotated intervals (valid + raw)
      - optionally parsed intervals

    If file does not exist or JSON structure is invalid -> returns None.
    """
    if not json_path.exists():
        return None

    try:
        with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
            data = json.load(f)
    except (json.JSONDecodeError, OSError):
        return None

    # --- flags ---
    raw_flags = data.get("flags")
    if raw_flags is None:
        raw_flags_list: List[str] = []
    elif isinstance(raw_flags, list):
        raw_flags_list = [str(x) for x in raw_flags]
    else:
        return None  # flags exists but is not a list -> invalid structure

    known_flags = known_flags or [
        "PSEUDO_ANNOTATED",
        "FOR_EXTERNAL_ANNOTATING",
        "EXTERNALLY_ANNOTATED",
        "FULLY_ANNOTATED_PROFESSIONALLY",
    ]

    raw_flags_set = set(raw_flags_list)
    flags_01 = {flag: (1 if flag in raw_flags_set else 0) for flag in known_flags}

    # --- noises_annotated ---
    items = data.get("noises_annotated")
    if items is None:
        raw_count = 0
        valid_intervals: List[Tuple[int, int]] = []
    elif isinstance(items, list):
        raw_count = len(items)
        valid_intervals = []
        for it in items:
            try:
                valid_intervals.append((int(it["startIndex"]), int(it["endIndex"])))
            except (KeyError, TypeError, ValueError):
                continue
    else:
        return None  # key exists but is not a list -> invalid structure

    result: EcgJsonSummary = {
        "flags_01": flags_01,
        "noises_annotated_count": len(valid_intervals),
        "noises_annotated_raw_count": raw_count,
        "intervals": valid_intervals if return_intervals else [],
    }
    return result


# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from project_util import find_project_root_by_name


# === Konfigūracija ==============================================================

PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

LIST_DIR = PROJECT_ROOT / PROJECT_DIR/ 'ecg_zive_npy_for_preparing'
REC_DIR = PROJECT_ROOT / "DATA_FOR_TRAINING"
# EXCEL_NAME = "visi_zive_irasai_atrankai_test.xlsx"
EXCEL_NAME = "visi_zive_irasai_atrankai.xlsx"

print("\nLIST_DIR:", LIST_DIR)
print("REC_DIR:", REC_DIR)
print("EXCEL_NAME:", EXCEL_NAME)

file_names, df_meta = read_filenames_from_excel(LIST_DIR / EXCEL_NAME)
# print(file_names)

# Normalizuojame meta failo pavadinimą iki "stem" (be plėtinio), kad galėtume tiksliai sulyginti.
df_meta = df_meta.copy()
df_meta["_stem"] = (
    df_meta["filename"].astype(str).str.strip().str.replace(r"\.(npy|json)$", "", regex=True)
)

flags_total = Counter()   # sums 0/1 across all records => "how many records have this flag"
records_with_noises = 0
records_without_noises = 0
total_records = 0

# optional diagnostics
missing_or_invalid = 0  # if your loader returns None for missing/invalid JSON

meta_matches = []  # list of df chunks to print once as a table at the end
missing_meta = []  # list of dicts for files not found in df_meta


for i, fname in enumerate(file_names, start=1):
    fpath = REC_DIR / fname

    try:
        result = count_flags_from_ecg_json(fpath.with_suffix(".json"))

        if result is not None:
            stem = fpath.stem
            rows = df_meta.loc[df_meta["_stem"] == stem].drop(columns=["_stem"], errors="ignore")
            # print(result)
            print_ecg_json_summary_row(result, prefix=f"{fpath.stem} | ")
            if rows.empty:
                missing_meta.append({"i": i, "fname": fname, "stem": stem})
            else:
                rows2 = rows.copy()
                rows2.insert(0, "i", i)
                rows2.insert(1, "fname", fname)
                meta_matches.append(rows2)



    except Exception as exc:
        raise ValueError(f"Failed to load {fname}: {exc}") from exc

    # If you kept "None means missing/invalid" semantics:
    if result is None:
        missing_or_invalid += 1
        continue

    total_records += 1

    # 1) aggregate flags
    flags_01: Dict[str, int] = result.get("flags_01", {})
    flags_total.update(flags_01)  # adds 0/1 per flag into totals

    # 2) aggregate noises_annotated presence
    n_noises = int(result.get("noises_annotated_count", 0))
    if n_noises > 0:
        records_with_noises += 1
    else:
        records_without_noises += 1


# ---- df_meta matches (print once) ----
if meta_matches:
    meta_all = pd.concat(meta_matches, ignore_index=True)

    # Use fixed widths (only applies in Jupyter display). Unknown columns will use default_px.
    _display_compact_table(
        meta_all,
        title="META (df_meta) rows matched by filename stem",
        col_widths_px=META_COL_WIDTHS_PX,
        default_px=80,
    )
else:
    print("=== META (df_meta) ===")
    print("No matching rows found in df_meta for the processed files.")

if missing_meta:
    miss_df = pd.DataFrame(missing_meta)
    _display_compact_table(
        miss_df,
        title="Missing df_meta rows for these files (by stem)",
        col_widths_px={"i": 45, "fname": 180, "stem": 130},
        default_px=120,
    )

# ---- summary printout ----
print("\n=== SUMMARY ===")
print(f"Processed records (valid JSON): {total_records}")
if missing_or_invalid:
    print(f"Skipped: {missing_or_invalid}")

print("\nFlags (how many records have each flag):")
for flag, cnt in sorted(flags_total.items()):
    print(f"  {flag}: {cnt}")

print("\nNoises annotated:")
print(f"  Records WITH noises_annotated intervals: {records_with_noises}")
print(f"  Records WITHOUT noises_annotated intervals: {records_without_noises}")



PROJECT ROOT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA

LIST_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/ecg_zive_npy_for_preparing
REC_DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/DATA_FOR_TRAINING
EXCEL_NAME: visi_zive_irasai_atrankai.xlsx

Skaitomas Excel: %s /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/1_PREPARE_TRAIN_UNET_DATA/ecg_zive_npy_for_preparing/visi_zive_irasai_atrankai.xlsx
Įrašų sąraše: 1085
1001_6 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 1
1001_8 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 1
1004_0 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 6
1005_0 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 1
1005_4 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 1
1006_1 | FLAGS: FULLY_ANNOTATED_PROFESSIONALLY | noises_annotated_count: 1
1007_1 | 

,i,fname,filename,length,quality,noni,tag,mark,N,S,V,U,recordingId,basename,userId,comment
0,7,1001_6.npy,1001_6,128000,0,1,1111,NC,718,0,2,0,638d8f609e4d4309eebbb0f1,1670188.752,60a917b354352a3df86dc1f2,nan
1,9,1001_8.npy,1001_8,128000,0,1,1111,x,737,0,2,0,638d8f5d9e4d43301ebbb0df,1670183.201,60a917b354352a3df86dc1f2,nan
2,19,1004_0.npy,1004_0,127999,0,6,1111,x,610,2,0,0,613f59083d08d46e7acdce29,1630715.197,613b1c013d08d44862cdc8f2,"labai stabilus įrašas, be triukšmų"
3,22,1005_0.npy,1005_0,127999,0,1,1111,x,559,5,0,0,613f56f73d08d4485dcdcb87,1630733.908,613b1c6f3d08d4370acdc8f3,"labai stabilus įrašas, be triukšmų, 5 S, nei vieno nerado"
4,34,1005_4.npy,1005_4,127999,0,1,1111,x,540,1,1,0,613f56f73d08d4bc97cdcb8a,1630736.382,613b1c6f3d08d4370acdc8f3,Labai švarus
5,41,1006_1.npy,1006_1,127999,0,1,1111,NC,580,0,1,0,613f59533d08d413facdcebb,1630797.676,613b1c9c3d08d43181cdc8f4,"labai stabilus įrašas, be triukšmų, 1 V, rado"
6,45,1007_1.npy,1007_1,127999,0,2,1111,x,612,0,0,0,613b221f3d08d453a1cdc914,1630959.214,613b1cc33d08d4c0d5cdc8f5,Labai geras
7,46,1008_0.npy,1008_0,127999,0,1,1111,x,724,2,21,0,613f58593d08d4cec1cdccf1,1630771.589,613b1d0c3d08d413ffcdc8f6,Gana silpnas BW
8,50,1008_12.npy,1008_12,127999,0,5,1111,x,708,4,5,0,613f58593d08d461c8cdcce8,1630758.549,613b1d0c3d08d413ffcdc8f6,"Nedidelis BW, ryškesnis įrašo pabaigoje"
9,75,1009_12.npy,1009_12,127999,0,2,1111,x,776,3,0,0,613b22d43d08d443f5cdc98e,1631057.107,613b1d673d08d4d1f3cdc8f8,Idealus



=== SUMMARY ===
Processed records (valid JSON): 35
Skipped: 1050

Flags (how many records have each flag):
  EXTERNALLY_ANNOTATED: 6
  FOR_EXTERNAL_ANNOTATING: 6
  FULLY_ANNOTATED_PROFESSIONALLY: 29
  PSEUDO_ANNOTATED: 0

Noises annotated:
  Records WITH noises_annotated intervals: 34
  Records WITHOUT noises_annotated intervals: 1
